In [ ]:
!pip install -q U bitsandbytes>=0.46.1 trl peft transformers datasets wandb


In [ ]:
import os
import re
import torch
from datasets import load_dataset
from peft import LoraConfig, PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from trl import GRPOConfig, GRPOTrainer

os.environ["WANDB_PROJECT"] = "gemma3-countdown-distillation"

MODEL_ID = "sft-2500"
DATASET_PATH = "dataset_3.jsonl"
OUTPUT_DIR = "gemma-grpo-countdown"
SEED = 1337


def accuracy_reward_func(prompts, completions, target, nums, **kwargs):
    rewards = []
    for completion, tgt, allowed_nums in zip(completions, target, nums):
        try:
            match = re.search(r"<answer>(.*?)</answer>", completion, re.DOTALL)
            if not match:
                rewards.append(0.0)
                continue

            equation = match.group(1).strip()
            if "=" in equation:
                equation = equation.split("=")[0].strip()

            used_nums = [int(n) for n in re.findall(r'\d+', equation)]

            if sorted(used_nums) != sorted(allowed_nums):
                rewards.append(0.0)
                continue

            clean_eq = re.sub(r'[^0-9+\-*/(). ]', '', equation)
            result = eval(clean_eq, {"__builtins__": None}, {})

            if abs(result - float(tgt)) < 1e-5:
                rewards.append(1.0)
            else:
                rewards.append(0.0)
        except Exception:
            rewards.append(0.0)
    return rewards

def format_reward_func(completions, **kwargs):
    rewards = []
    for completion in completions:
        pattern = r"^<think>.*?</think>\s*<answer>.*?</answer>$"
        if re.search(pattern, completion, re.DOTALL):
            rewards.append(0.2)
        else:
            rewards.append(0.0)
    return rewards

def penalty_reward_func(completions, **kwargs):
    rewards = []
    for completion in completions:
        penalty = 0.0
        low_c = completion.lower()

        if any(word in low_c for word in ["wait", "retry", "let's try", "maybe"]):
            penalty -= 0.2

        think_match = re.search(r"<think>(.*?)</think>", completion, re.DOTALL)
        if think_match:
            if len(think_match.group(1)) > 1000:
                penalty -= 0.3
        else:
            penalty -= 0.1

        rewards.append(penalty)
    return rewards


def prepare_data(example):
    messages = example["messages"]
    prompt = f"<start_of_turn>system\n{messages[0]['content']}<end_of_turn>\n"
    prompt += f"<start_of_turn>user\n{messages[1]['content']}<end_of_turn>\n<start_of_turn>model\n"

    user_text = messages[1]['content']
    target = int(re.search(r"equals (\d+)", user_text).group(1))
    nums_match = re.search(r"\[(.*?)\]", user_text)
    nums = [int(x.strip()) for x in nums_match.group(1).split(',')]

    return {
        "prompt": prompt,
        "target": target,
        "nums": nums
    }


dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
dataset = dataset.shuffle(seed=SEED).map(prepare_data)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

BASE_MODEL_ID = "google/gemma-3-1b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

model = PeftModel.from_pretrained(model, MODEL_ID, is_trainable=True)

lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

training_args = GRPOConfig(
    output_dir=OUTPUT_DIR,
    learning_rate=5e-6,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    max_completion_length=1280,
    num_generations=4,
    bf16=True,
    num_train_epochs=1,
    run_name="distill-gemma1b-grpo",
    save_steps=30,
    logging_steps=1,
    report_to="wandb",
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[accuracy_reward_func, format_reward_func, penalty_reward_func],
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

trainer.train()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'pad_token_id': 1}.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: nokish (nokish-digital-inc) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
1,0.000000
2,0.015755
3,0.098466
4,0.006839
5,-0.005258
6,0.124002
7,0.026402
8,0.142605
9,-0.068847
10,0.222503


KeyboardInterrupt: 